<a href="https://www.kaggle.com/code/avikdas567/digitization-of-ecg-images?scriptVersionId=291479917" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet
/kaggle/input/physionet-ecg-image-digitization/train.csv
/kaggle/input/physionet-ecg-image-digitization/test.csv
/kaggle/input/physionet-ecg-image-digitization/test/2352854581.png
/kaggle/input/physionet-ecg-image-digitization/test/1053922973.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0005.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0006.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0011.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0004.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893.csv
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0012.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0003.png
/kaggle/input/physionet-ecg-image-digitization/train/735384893/735384893-0001.png
/kaggle/input/physionet-ecg-i

# Imports & Config

In [2]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load Test Metadata

In [3]:
TEST_CSV = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_SUB = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"

test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_parquet(SAMPLE_SUB)

print("Test rows:", len(test_df))
print("Submission rows:", len(sample_sub))

Test rows: 24
Submission rows: 75000


# ECG Length Helper

In [4]:
def expected_length(fs, lead):
    """
    Lead II: 10 seconds
    Others: 2.5 seconds
    """
    if lead == "II":
        return int(np.floor(fs * 10.0))
    else:
        return int(np.floor(fs * 2.5))

# Baseline ECG Generator

In [5]:
def generate_baseline_ecg(length, fs):
    t = np.arange(length) / fs
    
    # tiny sinusoidal prior (~heart rhythm)
    signal = 0.01 * np.sin(2 * np.pi * 1.2 * t)
    
    # very small noise
    noise = 0.002 * np.random.randn(length)
    
    return signal + noise

# Generate Predictions

In [6]:
records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    base_id = row["id"]
    fs = row["fs"]
    lead = row["lead"]
    
    length = expected_length(fs, lead)
    signal = generate_baseline_ecg(length, fs)
    
    for i, val in enumerate(signal):
        records.append({
            "id": f"{base_id}_{i}_{lead}",
            "value": float(val)
        })

100%|██████████| 24/24 [00:00<00:00, 217.19it/s]


# Create Submission File

In [7]:
submission = pd.DataFrame(records)

# Ensure ordering matches sample
submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()

submission.head()

,id,value
0,1053922973_0_I,0.001526
1,1053922973_1_I,0.002485
2,1053922973_2_I,0.002999
3,1053922973_3_I,-0.004004
4,1053922973_4_I,0.001388


# Save Submission

In [8]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Rows:", len(submission))

Saved: submission.csv
Rows: 75000
